In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("./dfs/final_df.csv")
df.sample(3)

,question,evidence,ground_truth,url,source
0,What is the difference between the MOMA datase...,"['Comparison with MOMA [ 20 ] .', 'First of al...",MOMA-LRG contains an order of magnitude more a...,https://openreview.net/forum?id=eJhc_CPXQIT,3
12,How does the arborescence-based training model...,"[""Graph-based Dissimilarity Let Gbe a graph wi...",The model calculates dissimilarity using a min...,https://aclanthology.org/2022.naacl-main.343.pdf,1
16,How does the mathematical application of the p...,"[""3.5 Positional Encoding\nSince our model con...",Vaswani et al. establish an additive positiona...,attention is all you need & RoPE,2


## Print Query Hierarchy Chunks Retrieval

In [2]:
import importlib
import main

importlib.reload(main)
run_hierarchy_pipeline = main.run_hierarchy_pipeline

test_query = "What is the difference between the MOMA dataset and the MOMA-LRG dataset?"
answer, hierarchy_template, retrieved_content, decision = run_hierarchy_pipeline(query=test_query, vector_db_path="./outputs/test3/", top_k=1)

Tree PNG saved -> decompositions\test6\decomposition_What_is_the_differen_e9d8d6ae-f53b-427e-858a-72d347219869.png


In [3]:
decision

RoutingDecision(route='MULTI_HOP', confidence=0.72, reason='Requires comparing two datasets and their characteristics.')

In [4]:
answer

'The MOMA-LRG dataset is a newer, larger, and more structured successor to the MOMA dataset, with several key differences:\n\n- Hierarchical abstraction: MOMA-LRG introduces activity graphs that span three levels of hierarchy—activities, sub-activities, and atomic actions (which are described by entities and predicates). In contrast, MOMA is described as using an abstraction limited to the atomic level. This makes MOMA-LRG a richer, multi-level representation of human activity.\n\n- Scale and diversity: MOMA-LRG contains order-of-magnitude more annotations and uses longer videos from a greater variety of scenes than MOMA.\n\n- Grounding and role labeling: MOMA-LRG grounds all associated entities, disambiguates which entities are involved in each interaction, and classifies each actor’s role, providing richer semantic grounding than what MOMA offers.\n\n- Dynamic vs. static predicates: MOMA-LRG differentiates between static and dynamic predicates (e.g., dynamic state transitions like si

In [5]:
print(hierarchy_template)

1. What is the difference between the MOMA dataset and the MOMA-LRG dataset?
1.1. What domains does MOMA cover?
        retrieved content:
          retrieved chunk: ![](images/03842729bcbf1edb4fd207a389ff9234bb9bbbad6c39f1e66934cd5c57af0c1d.jpg)  
Figure 2: Partonomic and taxonomic hierarchies of MOMA-LRG. MOMA-LRG breaks down activities into sub-activities, which are in turn described by atomic actions. Atomic actions are broken down into entities (actors and objects), whose interactions with each other are described by predicates that either attributes (unary, involving one entity) or relationship (binary, involving two entities).

action interaction instances, which can be further broken down into actors, objects, relationships, and attributes. Specifically, MOMA-LRG contains

• 104,564 actor instances (636,194 bounding boxes) from 26 classes;   
• 47,494 object instances (349,034 bounding boxes) from 225 unique classes;   
• 1,037,319 relationships from 52 classes;   
• 704,230 re

In [6]:
retrieved_content

['![](images/03842729bcbf1edb4fd207a389ff9234bb9bbbad6c39f1e66934cd5c57af0c1d.jpg)  \nFigure 2: Partonomic and taxonomic hierarchies of MOMA-LRG. MOMA-LRG breaks down activities into sub-activities, which are in turn described by atomic actions. Atomic actions are broken down into entities (actors and objects), whose interactions with each other are described by predicates that either attributes (unary, involving one entity) or relationship (binary, involving two entities).\n\naction interaction instances, which can be further broken down into actors, objects, relationships, and attributes. Specifically, MOMA-LRG contains\n\n• 104,564 actor instances (636,194 bounding boxes) from 26 classes;   \n• 47,494 object instances (349,034 bounding boxes) from 225 unique classes;   \n• 1,037,319 relationships from 52 classes;   \n• 704,230 relationships from 13 classes.\n\nLanguage-Refined Graphs. One of MOMA-LRG’s distinguishing features is that it enables few-shot capabilities. To do this, we 

## Querying

In [17]:
from datetime import datetime

db_dir = {
    "1": "./outputs/test1/",
    "2": "./outputs/test2/",
    "3": "./outputs/test3/"
}

In [18]:
start = datetime.now()
all_answers = []
qa_latencies = []

for db_id, input_dir in db_dir.items():
    print(f"Running pipeline for DB {db_id}...")
    rows = df[df["source"] == int(db_id)]

    for _, row in rows.iterrows():
        question = row["question"]
        print(f"Question: {question}")
        s = datetime.now()
        answer, hierarchy_template, retrieved_content, decision = run_hierarchy_pipeline(query=question, vector_db_path=input_dir, top_k=3)
        print(f"Time taken for question: {(datetime.now() - s).seconds} seconds")
        qa_latencies.append((datetime.now() - s).seconds)
        unique_docs = set(retrieved_content)
        all_answers.append({
                "question": question,
                "answer": answer,
                "context": unique_docs,
                "ground_truth": row["ground_truth"],
                "evidence": row["evidence"],
                "source": db_id
            })
end = datetime.now()
ellapsed_time = end - start
print(f"Total time taken: {ellapsed_time}")

Running pipeline for DB 1...
Question: What is the target KB size used for MedMentions in the experiments?
Time taken for question: 9 seconds
Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Tree PNG saved -> decompositions\test6\decomposition_Do_the_training_and__67784154-7d23-44a2-9447-1e7c845471e4.png
Time taken for question: 39 seconds
Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Time taken for question: 21 seconds
Question: Why not use the complete graph for training instead of pruning it?
Tree PNG saved -> decompositions\test6\decomposition_Why_not_use_the_comp_b1b7fff0-69ce-425c-a428-b1aedd43d1d4.png
Time taken for question: 56 seconds
Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for t

In [20]:
max(qa_latencies), min(qa_latencies), sum(qa_latencies)/len(qa_latencies)

(68, 9, 42.36842105263158)

In [19]:
qa_latencies

[9, 39, 21, 56, 47, 62, 47, 39, 57, 32, 58, 43, 35, 45, 68, 43, 62, 25, 17]

In [3]:
all_answers_df = pd.DataFrame(all_answers)
all_answers_df.to_csv("all_answers_hierarchy_final.csv", index=False)

NameError: name 'pd' is not defined

## Evaluation

In [22]:
def f1_token_overlap(pred, truth):
    pred_tokens = set(pred.lower().split())
    truth_tokens = set(truth.lower().split())
    if not pred_tokens or not truth_tokens:
        return 0
    return 2 * len(pred_tokens & truth_tokens) / (len(pred_tokens) + len(truth_tokens))

all_answers_df['f1'] = all_answers_df.apply(
    lambda row: f1_token_overlap(row['answer'], row['ground_truth']), axis=1
)

In [2]:
all_answers_df.sample(2)

NameError: name 'all_answers_df' is not defined

In [1]:
df_with_eval = all_answers_df.copy()

NameError: name 'all_answers_df' is not defined

In [ ]:
from rag_eval import evaluate

results = []
for row in all_answers_df.itertuples():
    print(f"Evaluating Question: {row.question}")
    result = evaluate(
        question=row.question,
        answer=row.answer,
        contexts= row.context,
        ground_truth=row.ground_truth
    )
    results.append(result)

Evaluating Question: What is the target KB size used for MedMentions in the experiments?
Evaluating Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Evaluating Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Evaluating Question: Why not use the complete graph for training instead of pruning it?
Evaluating Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for this calculation?
Evaluating Question: Why do the proposed arborescence-based cross-encoder models show consistent gains in linking accuracy on the MedMentions dataset but mixed results compared to mention-entity baselines on the ZeShEL dataset, and what specific statistical difference accounts for this?
Evaluating Question: During the constraine

In [35]:
for i, result in enumerate(results):
    for metric_name, metric_value in result.items():
        if isinstance(metric_value, dict):
            df_with_eval.at[i, metric_name] = metric_value.get("score")
        else:
            df_with_eval.at[i, metric_name] = metric_value

In [27]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          0.87500
context_relevance     0.82500
answer_relevance      1.00000
answer_correctness    0.87500
overall               0.89375
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.916667
context_relevance     0.833333
answer_relevance      0.933333
answer_correctness    0.833333
overall               0.879167
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          1.000000
context_relevance     0.550000
answer_relevance      1.000000
answer_correctness    0.437500
overall               0.746875
dtype: float64


In [30]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          0.7500
context_relevance     0.8250
answer_relevance      1.0000
answer_correctness    0.8750
overall               0.8625
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.916667
context_relevance     0.866667
answer_relevance      0.966667
answer_correctness    0.833333
overall               0.895833
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          1.0000
context_relevance     0.5500
answer_relevance      1.0000
answer_correctness    0.5000
overall               0.7625
dtype: float64


In [36]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          0.875000
context_relevance     0.812500
answer_relevance      1.000000
answer_correctness    0.875000
overall               0.890625
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          0.916667
context_relevance     0.866667
answer_relevance      0.933333
answer_correctness    0.833333
overall               0.887500
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          1.000000
context_relevance     0.550000
answer_relevance      1.000000
answer_correctness    0.437500
overall               0.746875
dtype: float64


In [37]:
df_with_eval.to_csv("all_answers_hierarchy_final_with_eval.csv", index=False)